In [1]:
from google.colab import drive
drive.mount('/content/drive')

import os, sys

REPO_URL = "https://github.com/slimani-meriem/w2vec.git"
REPO_DIR = "/content/w2vec"

if not os.path.exists(REPO_DIR):
    !git clone {REPO_URL} {REPO_DIR}
else:
    !git -C {REPO_DIR} pull

sys.path.insert(0, f"{REPO_DIR}/src")

from data import *
from w2v import train_from_corpus
from eval_w2v import get_embeddings, evaluate, evaluate_runs

Mounted at /content/drive
Cloning into '/content/w2vec'...
remote: Enumerating objects: 32, done.
remote: Counting objects: 100% (32/32), done.
remote: Compressing objects: 100% (25/25), done.
remote: Total 32 (delta 8), reused 23 (delta 5), pack-reused 0 (from 0)
Receiving objects: 100% (32/32), 376.74 KiB | 1020.00 KiB/s, done.
Resolving deltas: 100% (8/8), done.


In [ ]:
import torch
print(torch.cuda.is_available(), torch.cuda.get_device_name(0) if torch.cuda.is_available() else "")

True Tesla T4


In [ ]:
!ls -lh /content/drive/MyDrive/tp-ml-w2vec

total 133M
-rw------- 1 root root 133M Nov  1 08:53 dataset.json


In [2]:
import json, time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from data import filter_rare_words, load_dataset
from w2v import w2v, save_embeddings
from eval_w2v import get_embeddings, evaluate

# --- Entrées (dépôt) ---
CORPUS   = f"{REPO_DIR}/data/Le_comte_de_Monte_Cristo.tok"
SIM_FILE = f"{REPO_DIR}/data/Le_comte_de_Monte_Cristo.100.sim.txt"

# --- Sorties (Drive, persistantes) ---
OUT_DIR      = "/content/drive/MyDrive/tp-ml-w2vec"
FILTERED     = f"{OUT_DIR}/filtered_text.txt"
DATASET_PATH = f"{OUT_DIR}/dataset.json"
RESULTS_CSV  = f"{OUT_DIR}/results.csv"
LOSSES_JSON  = f"{OUT_DIR}/losses.json"

# --- Hyperparamètres de l'énoncé ---
N = 100        # dimension des embeddings
ETA = 0.1      # taux d'apprentissage
E = 5          # nombre d'itérations
MINC = 5       # occurrences minimales
N_RUNS = 10

In [3]:
_, valid_words = filter_rare_words(CORPUS, FILTERED, minc=MINC)
dataset = load_dataset(DATASET_PATH)

print(f"Vocabulaire : {len(valid_words)} mots")
print(f"Dataset : {len(dataset)} exemples")

Vocabulaire : 3156 mots
Dataset : 876832 exemples


In [4]:
# test d'exécution
t0 = time.time()
W_test, losses_test = w2v(dataset, valid_words, n=N, max_iter=1, eta=ETA, seed=0)
dt = time.time() - t0
print(f"1 itération : {dt:.0f} s")
print(f"Estimation pour {N_RUNS} runs x {E} itérations : {dt * E * N_RUNS / 3600:.1f} h")

Itération 1/1 - perte moyenne : 3.0534
1 itération : 70 s
Estimation pour 10 runs x 5 itérations : 1.0 h


In [5]:
rows = []
all_losses = {}

for seed in range(N_RUNS):          # changer la seed pour changer l'initialisation de W et des permutations dans les données d'entrainement
    out = f"{OUT_DIR}/embeddings_seed{seed}.txt"
    t0 = time.time()

    W, losses = w2v(dataset, valid_words, n=N, max_iter=E, eta=ETA, seed=seed)
    save_embeddings(W, valid_words, out)
    res = evaluate(get_embeddings(out), SIM_FILE)

    rows.append({"seed": seed, "rate": res["rate"], "correct": res["correct"],
                 "total": res["total"], "oov": res["oov"],
                 "final_loss": losses[-1], "time_s": round(time.time() - t0)})
    all_losses[str(seed)] = losses

    print(f"seed {seed} : taux = {res['rate']:.3f} | perte finale = {losses[-1]:.3f} "
          f"| {rows[-1]['time_s']} s")

pd.DataFrame(rows).to_csv(RESULTS_CSV, index=False)
json.dump(all_losses, open(LOSSES_JSON, "w"))

Itération 1/5 - perte moyenne : 3.0534
Itération 2/5 - perte moyenne : 2.8751
Itération 3/5 - perte moyenne : 2.8358
Itération 4/5 - perte moyenne : 2.8190
Itération 5/5 - perte moyenne : 2.8111
seed 0 : taux = 0.760 | perte finale = 2.811 | 311 s
Itération 1/5 - perte moyenne : 3.0540
Itération 2/5 - perte moyenne : 2.8746
Itération 3/5 - perte moyenne : 2.8364
Itération 4/5 - perte moyenne : 2.8194
Itération 5/5 - perte moyenne : 2.8109
seed 1 : taux = 0.780 | perte finale = 2.811 | 308 s
Itération 1/5 - perte moyenne : 3.0541
Itération 2/5 - perte moyenne : 2.8747
Itération 3/5 - perte moyenne : 2.8357
Itération 4/5 - perte moyenne : 2.8198
Itération 5/5 - perte moyenne : 2.8104
seed 2 : taux = 0.760 | perte finale = 2.810 | 312 s
Itération 1/5 - perte moyenne : 3.0547
Itération 2/5 - perte moyenne : 2.8746
Itération 3/5 - perte moyenne : 2.8363
Itération 4/5 - perte moyenne : 2.8188
Itération 5/5 - perte moyenne : 2.8114
seed 3 : taux = 0.690 | perte finale = 2.811 | 310 s
Itératio

In [ ]:
# essayer d'expliquer la raison pour laquelle les loss sont élevés
rows = []
all_losses = {}

for seed in range(N_RUNS):          # changer la seed pour changer l'initialisation de W et des permutations dans les données d'entrainement
    out = f"{OUT_DIR}/embeddings_seed{seed}.txt"
    t0 = time.time()

    W, losses = w2v(dataset, valid_words, n=N, max_iter=E, eta=ETA, seed=seed)
    # --------------------------------------------------------
    loss_pos = -np.log(p_pos + 1e-10)
    loss_neg = -np.sum(np.log(1.0 - p_neg + 1e-10))
    total_pos += loss_pos
    total_neg += loss_neg
    # --------------------------------------------------------
    save_embeddings(W, valid_words, out)
    res = evaluate(get_embeddings(out), SIM_FILE)

    rows.append({"seed": seed, "rate": res["rate"], "correct": res["correct"],
                 "total": res["total"], "oov": res["oov"],
                 "final_loss": losses[-1], "time_s": round(time.time() - t0)})
    all_losses[str(seed)] = losses

    print(f"seed {seed} : taux = {res['rate']:.3f} | perte finale = {losses[-1]:.3f} "
          f"| {rows[-1]['time_s']} s")

pd.DataFrame(rows).to_csv(RESULTS_CSV, index=False)
json.dump(all_losses, open(LOSSES_JSON, "w"))

In [ ]:
!cp /content/w2vec/data/filtered_text.txt /content/drive/MyDrive/tp-ml-w2vec//